In [2]:
# ============================================================
# DIGITAL SHADOW — DEVELOPER PRODUCTIVITY INTELLIGENCE
# COMPLETE ONE-CELL COLAB VERSION
# ============================================================

import sys
import subprocess
import os
import textwrap
import time

# ------------------------------------------------------------
# 1. INSTALL PACKAGES
# ------------------------------------------------------------

packages = [
    "streamlit",
    "pandas",
    "numpy",
    "plotly",
    "scikit-learn",
    "networkx",
    "GitPython"
]

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q"
] + packages)

# ------------------------------------------------------------
# 2. COMPLETE STREAMLIT APPLICATION
# ------------------------------------------------------------

APP_CODE = r'''
import os
import io
import json
import math
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import networkx as nx
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go

from sklearn.ensemble import IsolationForest
from git import Repo


# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="Digital Shadow",
    page_icon="DS",
    layout="wide",
    initial_sidebar_state="expanded"
)


# ============================================================
# CUSTOM CSS
# ============================================================

st.markdown("""
<style>

html, body, [class*="css"] {
    font-family: Inter, Arial, sans-serif;
}

.stApp {
    background:
        radial-gradient(circle at 10% 10%, rgba(90,90,255,0.15), transparent 25%),
        radial-gradient(circle at 90% 10%, rgba(0,220,180,0.10), transparent 25%),
        linear-gradient(135deg, #05070d, #0b0f19 50%, #06080d);
    color: #f5f7fb;
}

.block-container {
    padding-top: 2rem;
    padding-bottom: 4rem;
    max-width: 1450px;
}

section[data-testid="stSidebar"] {
    background: #070a12;
    border-right: 1px solid rgba(255,255,255,0.08);
}

.hero {
    padding: 35px;
    border-radius: 25px;
    background:
        linear-gradient(
            135deg,
            rgba(255,255,255,0.08),
            rgba(255,255,255,0.025)
        );
    border: 1px solid rgba(255,255,255,0.10);
    box-shadow: 0 20px 60px rgba(0,0,0,0.35);
    margin-bottom: 25px;
}

.hero-title {
    font-size: 48px;
    font-weight: 800;
    letter-spacing: -2px;
    margin-bottom: 5px;
}

.hero-subtitle {
    color: #9ca8bc;
    font-size: 18px;
}

.metric-card {
    padding: 20px;
    border-radius: 18px;
    background: rgba(255,255,255,0.045);
    border: 1px solid rgba(255,255,255,0.09);
    min-height: 120px;
}

.metric-label {
    color: #8f9bb0;
    font-size: 13px;
    text-transform: uppercase;
    letter-spacing: 1px;
}

.metric-value {
    font-size: 32px;
    font-weight: 800;
    margin-top: 8px;
}

.section-title {
    font-size: 27px;
    font-weight: 750;
    margin-top: 25px;
    margin-bottom: 10px;
}

.insight-card {
    padding: 18px;
    margin: 10px 0;
    border-radius: 16px;
    background: rgba(255,255,255,0.04);
    border: 1px solid rgba(255,255,255,0.08);
}

.small-muted {
    color: #8d98aa;
    font-size: 13px;
}

.badge {
    display: inline-block;
    padding: 5px 10px;
    border-radius: 20px;
    background: rgba(100,120,255,0.15);
    border: 1px solid rgba(100,120,255,0.3);
    margin-right: 5px;
    font-size: 12px;
}

</style>
""", unsafe_allow_html=True)


# ============================================================
# SESSION STATE
# ============================================================

if "commits" not in st.session_state:
    st.session_state.commits = None

if "files" not in st.session_state:
    st.session_state.files = None

if "repo_name" not in st.session_state:
    st.session_state.repo_name = ""

if "analyzed" not in st.session_state:
    st.session_state.analyzed = False


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def safe_int(value):
    try:
        return int(value)
    except Exception:
        return 0


def safe_float(value):
    try:
        return float(value)
    except Exception:
        return 0.0


def get_extension(filename):
    filename = str(filename)

    if "." not in filename:
        return "no_extension"

    return filename.rsplit(".", 1)[-1].lower()


def short_hash(value):
    value = str(value)
    return value[:8]


# ============================================================
# GIT REPOSITORY COLLECTION
# ============================================================

def collect_repository(repo_path):

    repo = Repo(repo_path)

    if repo.bare:
        raise ValueError("The selected repository is bare.")

    commits_data = []
    files_data = []

    commits = list(repo.iter_commits("--all"))

    for commit in commits:

        commit_hash = commit.hexsha

        author = (
            commit.author.name
            if commit.author
            else "Unknown"
        )

        email = (
            commit.author.email
            if commit.author
            else ""
        )

        commit_date = datetime.fromtimestamp(
            commit.committed_date
        )

        message = (
            commit.message.strip().split("\n")[0]
            if commit.message
            else ""
        )

        insertions = 0
        deletions = 0
        files_changed = 0

        try:

            if commit.parents:

                parent = commit.parents[0]

                stats = commit.stats.total

                insertions = safe_int(
                    stats.get("insertions", 0)
                )

                deletions = safe_int(
                    stats.get("deletions", 0)
                )

                files_changed = safe_int(
                    stats.get("files", 0)
                )

                diff = parent.diff(
                    commit,
                    create_patch=False
                )

                for item in diff:

                    path = (
                        item.b_path
                        or item.a_path
                        or "unknown"
                    )

                    change_type = str(
                        item.change_type
                    )

                    files_data.append({
                        "commit_hash": commit_hash,
                        "short_hash": short_hash(commit_hash),
                        "author": author,
                        "email": email,
                        "date": commit_date,
                        "message": message,
                        "file": path,
                        "extension": get_extension(path),
                        "change_type": change_type
                    })

        except Exception:
            pass

        commits_data.append({
            "hash": commit_hash,
            "short_hash": short_hash(commit_hash),
            "author": author,
            "email": email,
            "date": commit_date,
            "message": message,
            "insertions": insertions,
            "deletions": deletions,
            "files_changed": files_changed
        })

    commits_df = pd.DataFrame(commits_data)
    files_df = pd.DataFrame(files_data)

    if not commits_df.empty:
        commits_df["date"] = pd.to_datetime(
            commits_df["date"]
        )

    if not files_df.empty:
        files_df["date"] = pd.to_datetime(
            files_df["date"]
        )

    return commits_df, files_df


# ============================================================
# DEVELOPER ANALYTICS
# ============================================================

def developer_analytics(commits, files):

    if commits.empty:
        return pd.DataFrame()

    result = (
        commits
        .groupby("author")
        .agg(
            commits=("hash", "count"),
            insertions=("insertions", "sum"),
            deletions=("deletions", "sum"),
            files_changed=("files_changed", "sum")
        )
        .reset_index()
    )

    result["total_changes"] = (
        result["insertions"] +
        result["deletions"]
    )

    result["avg_files_per_commit"] = (
        result["files_changed"] /
        result["commits"].replace(0, 1)
    ).round(2)

    if not files.empty:

        unique_files = (
            files
            .groupby("author")["file"]
            .nunique()
            .reset_index()
            .rename(
                columns={"file": "unique_files"}
            )
        )

        result = result.merge(
            unique_files,
            on="author",
            how="left"
        )

    else:
        result["unique_files"] = 0

    total = result["total_changes"].sum()

    if total > 0:
        result["activity_percentage"] = (
            result["total_changes"] / total * 100
        ).round(2)
    else:
        result["activity_percentage"] = 0

    result["specialization"] = np.where(
        result["unique_files"] <= 5,
        "Focused",
        np.where(
            result["unique_files"] <= 20,
            "Balanced",
            "Broad"
        )
    )

    return result.sort_values(
        "total_changes",
        ascending=False
    )


# ============================================================
# CODE GRAVITY
# ============================================================

def calculate_code_gravity(files):

    if files.empty:
        return pd.DataFrame()

    gravity = (
        files
        .groupby("file")
        .agg(
            changes=("file", "count"),
            contributors=("author", "nunique"),
            commits=("commit_hash", "nunique")
        )
        .reset_index()
    )

    if gravity["changes"].max() > 0:
        gravity["gravity_score"] = (
            gravity["changes"] /
            gravity["changes"].max() *
            100
        ).round(2)
    else:
        gravity["gravity_score"] = 0

    gravity["gravity_level"] = pd.cut(
        gravity["gravity_score"],
        bins=[-1, 25, 50, 75, 101],
        labels=[
            "Low",
            "Medium",
            "High",
            "Critical"
        ]
    )

    return gravity.sort_values(
        "gravity_score",
        ascending=False
    )


# ============================================================
# TEMPORAL ANALYSIS
# ============================================================

def temporal_analysis(commits):

    if commits.empty:
        return pd.DataFrame()

    daily = (
        commits
        .set_index("date")
        .resample("D")
        .agg(
            commits=("hash", "count"),
            developers=("author", "nunique"),
            insertions=("insertions", "sum"),
            deletions=("deletions", "sum")
        )
        .reset_index()
    )

    daily["total_changes"] = (
        daily["insertions"] +
        daily["deletions"]
    )

    daily["rolling_7"] = (
        daily["commits"]
        .rolling(7, min_periods=1)
        .mean()
    )

    daily["rolling_30"] = (
        daily["commits"]
        .rolling(30, min_periods=1)
        .mean()
    )

    return daily


# ============================================================
# FILE TYPE ANALYSIS
# ============================================================

def file_type_analysis(files):

    if files.empty:
        return pd.DataFrame()

    result = (
        files
        .groupby("extension")
        .agg(
            changes=("file", "count"),
            files=("file", "nunique"),
            contributors=("author", "nunique")
        )
        .reset_index()
    )

    result["percentage"] = (
        result["changes"] /
        result["changes"].sum() *
        100
    ).round(2)

    return result.sort_values(
        "changes",
        ascending=False
    )


# ============================================================
# ANOMALY DETECTION
# ============================================================

def detect_anomalies(daily):

    if daily.empty or len(daily) < 10:

        daily = daily.copy()

        if not daily.empty:
            daily["anomaly"] = 1
            daily["anomaly_score"] = 0

        return daily

    features = [
        "commits",
        "developers",
        "insertions",
        "deletions"
    ]

    X = daily[features].fillna(0)

    model = IsolationForest(
        n_estimators=250,
        contamination=0.05,
        random_state=42
    )

    prediction = model.fit_predict(X)
    score = model.decision_function(X)

    result = daily.copy()

    result["anomaly"] = prediction
    result["anomaly_score"] = score

    return result


# ============================================================
# CHANGE RIPPLE
# ============================================================

def calculate_change_ripple(files):

    if files.empty:
        return pd.DataFrame()

    records = []

    for commit_hash, group in files.groupby("commit_hash"):

        changed_files = list(
            group["file"]
            .dropna()
            .unique()
        )

        if len(changed_files) > 100:
            changed_files = changed_files[:100]

        for i in range(len(changed_files)):

            for j in range(i + 1, len(changed_files)):

                records.append({
                    "file_a": changed_files[i],
                    "file_b": changed_files[j],
                    "commit_hash": commit_hash
                })

    if not records:
        return pd.DataFrame(
            columns=[
                "file_a",
                "file_b",
                "co_changes"
            ]
        )

    ripple = pd.DataFrame(records)

    ripple = (
        ripple
        .groupby(["file_a", "file_b"])
        .size()
        .reset_index(name="co_changes")
        .sort_values(
            "co_changes",
            ascending=False
        )
    )

    return ripple


# ============================================================
# COLLABORATION NETWORK
# ============================================================

def build_collaboration_network(files):

    graph = nx.Graph()

    if files.empty:
        return graph

    for commit_hash, group in files.groupby("commit_hash"):

        developers = list(
            group["author"]
            .dropna()
            .unique()
        )

        for developer in developers:
            graph.add_node(developer)

        for i in range(len(developers)):

            for j in range(i + 1, len(developers)):

                a = developers[i]
                b = developers[j]

                if graph.has_edge(a, b):
                    graph[a][b]["weight"] += 1
                else:
                    graph.add_edge(
                        a,
                        b,
                        weight=1
                    )

    return graph


# ============================================================
# KNOWLEDGE CONCENTRATION
# ============================================================

def calculate_knowledge_concentration(files):

    if files.empty:
        return pd.DataFrame()

    result = (
        files
        .groupby("file")
        .agg(
            contributors=("author", "nunique"),
            changes=("file", "count")
        )
        .reset_index()
    )

    result["concentration_score"] = (
        1 /
        result["contributors"].replace(0, 1) *
        100
    ).round(2)

    result["risk"] = np.where(
        result["contributors"] == 1,
        "High",
        np.where(
            result["contributors"] == 2,
            "Medium",
            "Distributed"
        )
    )

    return result.sort_values(
        "concentration_score",
        ascending=False
    )


# ============================================================
# PROJECT HEALTH
# ============================================================

def calculate_project_health(
    commits,
    files,
    gravity,
    anomalies
):

    total_commits = len(commits)

    developers = (
        commits["author"].nunique()
        if not commits.empty
        else 0
    )

    activity_score = min(
        100,
        total_commits / 10
    )

    diversity_score = min(
        100,
        developers * 10
    )

    if not anomalies.empty:
        anomaly_ratio = (
            (anomalies["anomaly"] == -1).mean()
        )
    else:
        anomaly_ratio = 0

    stability_score = max(
        0,
        100 - anomaly_ratio * 100
    )

    if not gravity.empty:

        critical_ratio = (
            gravity["gravity_level"]
            .isin(["High", "Critical"])
            .mean()
        )

    else:
        critical_ratio = 0

    distribution_score = max(
        0,
        100 - critical_ratio * 100
    )

    overall = np.mean([
        activity_score,
        diversity_score,
        stability_score,
        distribution_score
    ])

    return {
        "overall": round(overall, 2),
        "activity": round(activity_score, 2),
        "diversity": round(diversity_score, 2),
        "stability": round(stability_score, 2),
        "distribution": round(distribution_score, 2)
    }


# ============================================================
# NETWORK FIGURE
# ============================================================

def create_network_figure(graph):

    if graph.number_of_nodes() == 0:

        fig = go.Figure()

        fig.update_layout(
            title="No collaboration data available"
        )

        return fig

    positions = nx.spring_layout(
        graph,
        seed=42,
        k=1.5
    )

    edge_x = []
    edge_y = []

    for a, b in graph.edges():

        x0, y0 = positions[a]
        x1, y1 = positions[b]

        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    edge_trace = go.Scatter(
        x=edge_x,
        y=edge_y,
        mode="lines",
        line=dict(
            width=1
        ),
        hoverinfo="none"
    )

    node_x = []
    node_y = []
    node_text = []
    node_size = []

    for node in graph.nodes():

        x, y = positions[node]

        node_x.append(x)
        node_y.append(y)

        degree = graph.degree(node)

        node_text.append(
            f"{node}<br>Connections: {degree}"
        )

        node_size.append(
            12 + degree * 5
        )

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers+text",
        text=list(graph.nodes()),
        textposition="top center",
        hovertext=node_text,
        hoverinfo="text",
        marker=dict(
            size=node_size
        )
    )

    fig = go.Figure(
        data=[
            edge_trace,
            node_trace
        ]
    )

    fig.update_layout(
        template="plotly_dark",
        height=600,
        showlegend=False,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        xaxis=dict(
            visible=False
        ),
        yaxis=dict(
            visible=False
        )
    )

    return fig


# ============================================================
# PROJECT INSIGHTS
# ============================================================

def generate_insights(
    commits,
    files,
    developers,
    gravity,
    daily,
    ripple,
    health
):

    insights = []

    if commits.empty:
        return insights

    insights.append(
        f"The repository contains {len(commits):,} commits "
        f"across {commits['author'].nunique()} contributors."
    )

    if not developers.empty:

        top = developers.iloc[0]

        insights.append(
            f"Most code activity comes from "
            f"{top['author']} with "
            f"{top['activity_percentage']:.1f}% "
            f"of recorded changes."
        )

    if not gravity.empty:

        top_file = gravity.iloc[0]

        insights.append(
            f"The strongest Code Gravity center is "
            f"{top_file['file']} with "
            f"{top_file['changes']} recorded changes."
        )

    if not daily.empty:

        peak = daily.loc[
            daily["commits"].idxmax()
        ]

        insights.append(
            f"The highest commit activity occurred on "
            f"{peak['date'].strftime('%Y-%m-%d')} "
            f"with {int(peak['commits'])} commits."
        )

    if not ripple.empty:

        top_ripple = ripple.iloc[0]

        insights.append(
            f"The strongest Change Ripple relationship is "
            f"{top_ripple['file_a']} <-> "
            f"{top_ripple['file_b']} "
            f"with {int(top_ripple['co_changes'])} "
            f"co-changes."
        )

    insights.append(
        f"Calculated Project Intelligence Health Score: "
        f"{health['overall']:.1f}/100."
    )

    return insights


# ============================================================
# REPORT GENERATOR
# ============================================================

def create_report(
    repo_name,
    commits,
    files,
    developers,
    gravity,
    daily,
    ripple,
    health,
    insights
):

    lines = []

    lines.append("=" * 70)
    lines.append("DIGITAL SHADOW")
    lines.append("Developer Productivity Intelligence Report")
    lines.append("=" * 70)
    lines.append("")

    lines.append(
        f"Repository: {repo_name}"
    )

    lines.append(
        f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
    )

    lines.append("")

    lines.append("PROJECT OVERVIEW")
    lines.append("-" * 40)

    lines.append(
        f"Commits: {len(commits)}"
    )

    lines.append(
        f"Developers: {commits['author'].nunique() if not commits.empty else 0}"
    )

    lines.append(
        f"Files touched: {files['file'].nunique() if not files.empty else 0}"
    )

    lines.append("")

    lines.append("PROJECT HEALTH")
    lines.append("-" * 40)

    for key, value in health.items():

        lines.append(
            f"{key.title()}: {value}"
        )

    lines.append("")

    lines.append("INTELLIGENCE")
    lines.append("-" * 40)

    for insight in insights:
        lines.append(
            "- " + insight
        )

    lines.append("")

    lines.append("TOP CODE GRAVITY FILES")
    lines.append("-" * 40)

    if not gravity.empty:

        for _, row in gravity.head(10).iterrows():

            lines.append(
                f"{row['file']} | "
                f"changes={row['changes']} | "
                f"contributors={row['contributors']}"
            )

    lines.append("")

    lines.append("TOP DEVELOPERS")
    lines.append("-" * 40)

    if not developers.empty:

        for _, row in developers.head(10).iterrows():

            lines.append(
                f"{row['author']} | "
                f"commits={row['commits']} | "
                f"changes={row['total_changes']}"
            )

    return "\n".join(lines)


# ============================================================
# HERO
# ============================================================

st.markdown("""
<div class="hero">

<div class="hero-title">
Digital Shadow
</div>

<div class="hero-subtitle">
Developer Productivity Intelligence Platform
</div>

<br>

<span class="badge">Git Intelligence</span>
<span class="badge">Developer DNA</span>
<span class="badge">Code Gravity</span>
<span class="badge">Anomaly Detection</span>
<span class="badge">Collaboration Graph</span>
<span class="badge">Project Intelligence</span>

</div>
""", unsafe_allow_html=True)


# ============================================================
# SIDEBAR
# ============================================================

with st.sidebar:

    st.markdown("## Digital Shadow")

    st.caption(
        "Turn Git history into engineering intelligence."
    )

    st.markdown("---")

    repo_path = st.text_input(
        "Repository Path",
        value=".",
        help="Use . for the current folder or provide an absolute path."
    )

    analyze = st.button(
        "Analyze Repository",
        use_container_width=True,
        type="primary"
    )

    st.markdown("---")

    st.markdown("### Analysis Pipeline")

    st.markdown("""
    1. Git History
    2. Commit Intelligence
    3. File Intelligence
    4. Developer DNA
    5. Code Gravity
    6. Project Evolution
    7. Anomaly Radar
    8. Change Ripple
    9. Collaboration Graph
    10. Knowledge Concentration
    11. Project Intelligence
    """)


# ============================================================
# ANALYZE
# ============================================================

if analyze:

    try:

        with st.spinner(
            "Analyzing Git repository..."
        ):

            commits, files = collect_repository(
                repo_path
            )

            st.session_state.commits = commits
            st.session_state.files = files
            st.session_state.repo_name = (
                Path(repo_path).resolve().name
            )
            st.session_state.analyzed = True

            output_dir = Path(
                "digital_shadow_output"
            )

            output_dir.mkdir(
                exist_ok=True
            )

            commits.to_csv(
                output_dir / "commits.csv",
                index=False
            )

            files.to_csv(
                output_dir / "file_changes.csv",
                index=False
            )

        st.success(
            "Repository analyzed successfully."
        )

    except Exception as e:

        st.error(
            f"Could not analyze repository: {e}"
        )

        st.stop()


# ============================================================
# CHECK DATA
# ============================================================

if not st.session_state.analyzed:

    st.info(
        "Enter a Git repository path in the sidebar and click "
        "'Analyze Repository'."
    )

    st.markdown("""
    ### What Digital Shadow analyzes

    **Commit Intelligence**
    - Commit frequency
    - Code additions/deletions
    - Developer activity

    **Developer DNA**
    - Individual contribution patterns
    - File specialization
    - Activity distribution

    **Code Gravity**
    - Most frequently changed files
    - Critical code centers
    - Contributor concentration

    **Project Evolution**
    - Activity over time
    - Development trends
    - Rolling activity

    **Anomaly Radar**
    - Unusual development days
    - Statistical activity anomalies

    **Change Ripple**
    - Files that repeatedly change together

    **Collaboration Graph**
    - Developer-to-developer collaboration

    **Knowledge Concentration**
    - Files controlled by few contributors

    **Project Intelligence**
    - Overall engineering health indicators
    """)

    st.stop()


# ============================================================
# LOAD DATA
# ============================================================

commits = st.session_state.commits
files = st.session_state.files

repo_name = st.session_state.repo_name


# ============================================================
# ANALYTICS
# ============================================================

developers = developer_analytics(
    commits,
    files
)

gravity = calculate_code_gravity(
    files
)

daily = temporal_analysis(
    commits
)

file_types = file_type_analysis(
    files
)

anomalies = detect_anomalies(
    daily
)

ripple = calculate_change_ripple(
    files
)

network = build_collaboration_network(
    files
)

knowledge = calculate_knowledge_concentration(
    files
)

health = calculate_project_health(
    commits,
    files,
    gravity,
    anomalies
)

insights = generate_insights(
    commits,
    files,
    developers,
    gravity,
    daily,
    ripple,
    health
)


# ============================================================
# NAVIGATION
# ============================================================

page = st.sidebar.radio(
    "Navigate",
    [
        "Command Center",
        "Code Gravity",
        "Developer DNA",
        "Project Evolution",
        "Anomaly Radar",
        "Change Ripple",
        "Collaboration Network",
        "Knowledge Concentration",
        "AI Insights",
        "Repository Explorer"
    ]
)


# ============================================================
# COMMAND CENTER
# ============================================================

if page == "Command Center":

    st.markdown(
        '<div class="section-title">Command Center</div>',
        unsafe_allow_html=True
    )

    c1, c2, c3, c4, c5 = st.columns(5)

    with c1:
        st.markdown(
            f"""
            <div class="metric-card">
            <div class="metric-label">Commits</div>
            <div class="metric-value">
            {len(commits):,}
            </div>
            </div>
            """,
            unsafe_allow_html=True
        )

    with c2:

        developers_count = (
            commits["author"].nunique()
            if not commits.empty
            else 0
        )

        st.markdown(
            f"""
            <div class="metric-card">
            <div class="metric-label">Developers</div>
            <div class="metric-value">
            {developers_count:,}
            </div>
            </div>
            """,
            unsafe_allow_html=True
        )

    with c3:

        files_count = (
            files["file"].nunique()
            if not files.empty
            else 0
        )

        st.markdown(
            f"""
            <div class="metric-card">
            <div class="metric-label">Files</div>
            <div class="metric-value">
            {files_count:,}
            </div>
            </div>
            """,
            unsafe_allow_html=True
        )

    with c4:

        total_changes = (
            commits["insertions"].sum() +
            commits["deletions"].sum()
        )

        st.markdown(
            f"""
            <div class="metric-card">
            <div class="metric-label">Code Changes</div>
            <div class="metric-value">
            {int(total_changes):,}
            </div>
            </div>
            """,
            unsafe_allow_html=True
        )

    with c5:

        st.markdown(
            f"""
            <div class="metric-card">
            <div class="metric-label">Health Score</div>
            <div class="metric-value">
            {health["overall"]:.1f}
            </div>
            </div>
            """,
            unsafe_allow_html=True
        )

    st.markdown("")

    if not daily.empty:

        fig = px.line(
            daily,
            x="date",
            y=[
                "commits",
                "rolling_7"
            ],
            title="Repository Activity"
        )

        fig.update_layout(
            template="plotly_dark",
            paper_bgcolor="rgba(0,0,0,0)",
            plot_bgcolor="rgba(0,0,0,0)"
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

    st.markdown(
        '<div class="section-title">Project Intelligence</div>',
        unsafe_allow_html=True
    )

    for insight in insights:

        st.markdown(
            f"""
            <div class="insight-card">
            {insight}
            </div>
            """,
            unsafe_allow_html=True
        )


# ============================================================
# CODE GRAVITY
# ============================================================

elif page == "Code Gravity":

    st.markdown(
        '<div class="section-title">Code Gravity</div>',
        unsafe_allow_html=True
    )

    st.write(
        "Code Gravity identifies the files that attract the "
        "largest amount of engineering activity."
    )

    if gravity.empty:

        st.warning(
            "No file-change information available."
        )

    else:

        top = gravity.head(20)

        fig = px.bar(
            top.sort_values(
                "gravity_score"
            ),
            x="gravity_score",
            y="file",
            orientation="h",
            title="Strongest Code Gravity Centers",
            hover_data=[
                "changes",
                "contributors",
                "commits"
            ]
        )

        fig.update_layout(
            template="plotly_dark",
            height=650,
            paper_bgcolor="rgba(0,0,0,0)",
            plot_bgcolor="rgba(0,0,0,0)"
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

        st.dataframe(
            gravity,
            use_container_width=True
        )


# ============================================================
# DEVELOPER DNA
# ============================================================

elif page == "Developer DNA":

    st.markdown(
        '<div class="section-title">Developer DNA</div>',
        unsafe_allow_html=True
    )

    if developers.empty:

        st.warning(
            "No developer information available."
        )

    else:

        c1, c2 = st.columns(2)

        with c1:

            fig = px.bar(
                developers.head(15),
                x="author",
                y="commits",
                title="Commits by Developer"
            )

            fig.update_layout(
                template="plotly_dark",
                xaxis_tickangle=-45
            )

            st.plotly_chart(
                fig,
                use_container_width=True
            )

        with c2:

            fig = px.bar(
                developers.head(15),
                x="author",
                y="total_changes",
                title="Code Changes by Developer"
            )

            fig.update_layout(
                template="plotly_dark",
                xaxis_tickangle=-45
            )

            st.plotly_chart(
                fig,
                use_container_width=True
            )

        fig = px.scatter(
            developers,
            x="commits",
            y="unique_files",
            size="total_changes",
            color="specialization",
            hover_name="author",
            title="Developer Breadth vs Activity"
        )

        fig.update_layout(
            template="plotly_dark"
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

        st.dataframe(
            developers,
            use_container_width=True
        )


# ============================================================
# PROJECT EVOLUTION
# ============================================================

elif page == "Project Evolution":

    st.markdown(
        '<div class="section-title">Project Evolution</div>',
        unsafe_allow_html=True
    )

    if daily.empty:

        st.warning(
            "No temporal information available."
        )

    else:

        fig = go.Figure()

        fig.add_trace(
            go.Scatter(
                x=daily["date"],
                y=daily["commits"],
                mode="lines+markers",
                name="Daily Commits"
            )
        )

        fig.add_trace(
            go.Scatter(
                x=daily["date"],
                y=daily["rolling_7"],
                mode="lines",
                name="7-Day Average"
            )
        )

        fig.update_layout(
            template="plotly_dark",
            title="Development Evolution",
            paper_bgcolor="rgba(0,0,0,0)",
            plot_bgcolor="rgba(0,0,0,0)"
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

        c1, c2 = st.columns(2)

        with c1:

            fig = px.area(
                daily,
                x="date",
                y="insertions",
                title="Code Insertions"
            )

            fig.update_layout(
                template="plotly_dark"
            )

            st.plotly_chart(
                fig,
                use_container_width=True
            )

        with c2:

            fig = px.area(
                daily,
                x="date",
                y="deletions",
                title="Code Deletions"
            )

            fig.update_layout(
                template="plotly_dark"
            )

            st.plotly_chart(
                fig,
                use_container_width=True
            )

        st.dataframe(
            daily,
            use_container_width=True
        )


# ============================================================
# ANOMALY RADAR
# ============================================================

elif page == "Anomaly Radar":

    st.markdown(
        '<div class="section-title">Anomaly Radar</div>',
        unsafe_allow_html=True
    )

    st.write(
        "Isolation Forest identifies unusually different "
        "development activity patterns."
    )

    if anomalies.empty:

        st.warning(
            "No anomaly information available."
        )

    else:

        anomaly_days = anomalies[
            anomalies["anomaly"] == -1
        ]

        st.metric(
            "Detected Anomalous Days",
            len(anomaly_days)
        )

        fig = px.scatter(
            anomalies,
            x="date",
            y="commits",
            size="total_changes",
            color="anomaly",
            title="Development Activity Anomalies",
            hover_data=[
                "developers",
                "insertions",
                "deletions"
            ]
        )

        fig.update_layout(
            template="plotly_dark"
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

        if not anomaly_days.empty:

            st.markdown(
                "### Anomalous Activity"
            )

            st.dataframe(
                anomaly_days,
                use_container_width=True
            )


# ============================================================
# CHANGE RIPPLE
# ============================================================

elif page == "Change Ripple":

    st.markdown(
        '<div class="section-title">Change Ripple</div>',
        unsafe_allow_html=True
    )

    st.write(
        "Change Ripple reveals files that repeatedly change "
        "together within the same commits."
    )

    if ripple.empty:

        st.warning(
            "No repeated file relationships found."
        )

    else:

        top = ripple.head(30).copy()

        top["relationship"] = (
            top["file_a"] +
            "  <->  " +
            top["file_b"]
        )

        fig = px.bar(
            top.sort_values(
                "co_changes"
            ),
            x="co_changes",
            y="relationship",
            orientation="h",
            title="Strongest File Relationships"
        )

        fig.update_layout(
            template="plotly_dark",
            height=700
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

        st.dataframe(
            ripple,
            use_container_width=True
        )


# ============================================================
# COLLABORATION NETWORK
# ============================================================

elif page == "Collaboration Network":

    st.markdown(
        '<div class="section-title">Collaboration Graph</div>',
        unsafe_allow_html=True
    )

    st.write(
        "This network shows developers who work on the same "
        "commits and files."
    )

    c1, c2, c3 = st.columns(3)

    with c1:

        st.metric(
            "Developers",
            network.number_of_nodes()
        )

    with c2:

        st.metric(
            "Collaboration Links",
            network.number_of_edges()
        )

    with c3:

        if network.number_of_nodes() > 0:

            density = nx.density(network)

        else:
            density = 0

        st.metric(
            "Network Density",
            f"{density:.3f}"
        )

    st.plotly_chart(
        create_network_figure(network),
        use_container_width=True
    )

    if network.number_of_nodes() > 0:

        degree_data = []

        for node in network.nodes():

            degree_data.append({
                "developer": node,
                "connections": network.degree(node)
            })

        degree_df = pd.DataFrame(
            degree_data
        ).sort_values(
            "connections",
            ascending=False
        )

        st.markdown(
            "### Collaboration Connectivity"
        )

        st.dataframe(
            degree_df,
            use_container_width=True
        )


# ============================================================
# KNOWLEDGE CONCENTRATION
# ============================================================

elif page == "Knowledge Concentration":

    st.markdown(
        '<div class="section-title">Knowledge Concentration</div>',
        unsafe_allow_html=True
    )

    st.write(
        "Files maintained by very few developers can represent "
        "knowledge concentration risk."
    )

    if knowledge.empty:

        st.warning(
            "No knowledge concentration data available."
        )

    else:

        top = knowledge.head(30)

        fig = px.bar(
            top.sort_values(
                "concentration_score"
            ),
            x="concentration_score",
            y="file",
            orientation="h",
            color="risk",
            title="Knowledge Concentration"
        )

        fig.update_layout(
            template="plotly_dark",
            height=750
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

        st.dataframe(
            knowledge,
            use_container_width=True
        )


# ============================================================
# AI INSIGHTS
# ============================================================

elif page == "AI Insights":

    st.markdown(
        '<div class="section-title">Project Intelligence</div>',
        unsafe_allow_html=True
    )

    st.markdown(
        """
        <div class="insight-card">
        Digital Shadow converts repository history into
        interpretable engineering intelligence.
        </div>
        """,
        unsafe_allow_html=True
    )

    c1, c2, c3, c4 = st.columns(4)

    with c1:
        st.metric(
            "Overall",
            f"{health['overall']:.1f}/100"
        )

    with c2:
        st.metric(
            "Activity",
            f"{health['activity']:.1f}/100"
        )

    with c3:
        st.metric(
            "Stability",
            f"{health['stability']:.1f}/100"
        )

    with c4:
        st.metric(
            "Distribution",
            f"{health['distribution']:.1f}/100"
        )

    st.markdown(
        "### Generated Intelligence"
    )

    for i, insight in enumerate(
        insights,
        start=1
    ):

        st.markdown(
            f"""
            <div class="insight-card">
            <strong>{i}.</strong> {insight}
            </div>
            """,
            unsafe_allow_html=True
        )

    report = create_report(
        repo_name,
        commits,
        files,
        developers,
        gravity,
        daily,
        ripple,
        health,
        insights
    )

    st.download_button(
        "Download Intelligence Report",
        data=report,
        file_name="digital_shadow_report.txt",
        mime="text/plain",
        use_container_width=True
    )


# ============================================================
# REPOSITORY EXPLORER
# ============================================================

elif page == "Repository Explorer":

    st.markdown(
        '<div class="section-title">Repository Explorer</div>',
        unsafe_allow_html=True
    )

    tab1, tab2, tab3 = st.tabs(
        [
            "Commits",
            "File Changes",
            "Developers"
        ]
    )

    with tab1:

        st.dataframe(
            commits.sort_values(
                "date",
                ascending=False
            ),
            use_container_width=True,
            height=600
        )

        csv = commits.to_csv(
            index=False
        ).encode()

        st.download_button(
            "Download Commits CSV",
            data=csv,
            file_name="commits.csv",
            mime="text/csv"
        )

    with tab2:

        st.dataframe(
            files.sort_values(
                "date",
                ascending=False
            ),
            use_container_width=True,
            height=600
        )

        csv = files.to_csv(
            index=False
        ).encode()

        st.download_button(
            "Download File Changes CSV",
            data=csv,
            file_name="file_changes.csv",
            mime="text/csv"
        )

    with tab3:

        st.dataframe(
            developers,
            use_container_width=True
        )

        csv = developers.to_csv(
            index=False
        ).encode()

        st.download_button(
            "Download Developer Analytics",
            data=csv,
            file_name="developer_analytics.csv",
            mime="text/csv"
        )


# ============================================================
# FOOTER
# ============================================================

st.markdown("---")

st.markdown(
    """
    <div style="text-align:center;color:#788397;padding:20px;">
    Digital Shadow — Developer Productivity Intelligence Platform
    <br>
    Git History → Intelligence → Engineering Insights
    </div>
    """,
    unsafe_allow_html=True
)
'''


# ------------------------------------------------------------
# 3. WRITE THE APP FILE
# ------------------------------------------------------------

app_path = "/content/digital_shadow.py"

with open(app_path, "w", encoding="utf-8") as f:
    f.write(APP_CODE)

print("Digital Shadow application created successfully.")
print("File:", app_path)

# ------------------------------------------------------------
# 4. START STREAMLIT
# ------------------------------------------------------------

import subprocess
import time
import os

# Stop any previous Streamlit process started by this notebook
subprocess.run(
    "pkill -f 'streamlit run digital_shadow.py' || true",
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        app_path,
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

print()
print("=" * 65)
print("DIGITAL SHADOW IS RUNNING")
print("=" * 65)
print()
print("Streamlit URL:")
print("http://localhost:8501")
print()
print("For Google Colab, use the next command if the URL is not")
print("directly accessible:")
print()
print("!npx localtunnel --port 8501")
print()
print("=" * 65)

Digital Shadow application created successfully.
File: /content/digital_shadow.py

DIGITAL SHADOW IS RUNNING

Streamlit URL:
http://localhost:8501

For Google Colab, use the next command if the URL is not
directly accessible:

!npx localtunnel --port 8501

